# IggyTop DB TCR anaylsis 

In [ ]:
import muon as mu
import scanpy as sc
import scirpy as ir

# temporary fix for deprecated matplotlib functionality
import IPython.display
from matplotlib_inline.backend_inline import set_matplotlib_formats

IPython.display.set_matplotlib_formats = set_matplotlib_formats

sc.set_figure_params(figsize=(4, 4))
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [ ]:
import numpy as np

In [ ]:
sc.logging.print_header()

## 1. Load data

In [ ]:
mdata = mu.read_h5mu("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/002_annotate_mudata_normal.h5mu")

In [ ]:
mdata

## 2. Creating chain indices & TCR QC

In [ ]:
ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)

In [ ]:
mu.pp.filter_obs(mdata, "airr:chain_pairing", lambda x: ~np.isin(x, ["orphan VDJ", "orphan VJ"]))

In [ ]:
# cells in AIRR that are TCR
tcr_cells = mdata["airr"].obs_names[
    mdata["airr"].obs["receptor_type"] == "TCR"
]

# keep only those cells in the full MuData object
mdata = mdata[tcr_cells, :].copy()

In [ ]:
mdata.update()

In [ ]:
mdata

## 3. Define clonotypes and clonotype clusters

In [ ]:
ir.pp.ir_dist(mdata)
ir.tl.define_clonotypes(mdata, receptor_arms="all", dual_ir="primary_only")

In [ ]:
ir.tl.clonotype_network(mdata, min_cells=5)

## 4. Top clones

In [ ]:
_ = ir.pl.group_abundance(mdata, groupby="airr:clone_id", target_col="gex:condition", max_cols=60, figsize=(20, 3))

### Top clonotypes filtered by threshold  > 100 cells 

In [ ]:
import pandas as pd

clone_sizes = (
    mdata.obs["airr:clone_id"]
    .value_counts()
    .rename_axis("clone_id")
    .reset_index(name="n_cells")
)

In [ ]:
large_clones = clone_sizes.loc[clone_sizes["n_cells"] > 10]

In [ ]:
abundance_df = pd.crosstab(
    mdata.obs["airr:clone_id"],
    mdata.obs["gex:condition"]
)

In [ ]:
clone_counts = abundance_df.sum(axis=1)

abundance_df_filtered = abundance_df.loc[clone_counts > 10]

In [ ]:
top_clone_ids = abundance_df_filtered.index.to_list()

In [ ]:
top_clone_ids

In [ ]:
import scanpy as sc

keep_clones = abundance_df_filtered.index

mdata_large = mdata[
    mdata.obs["airr:clone_id"].isin(keep_clones)
].copy()

In [ ]:
mdata_large

In [ ]:
ir.pl.group_abundance(
    mdata_large,
    groupby="airr:clone_id",
    target_col="gex:condition",
    figsize=(20, 5)
)

## 5. Iggytop analysis DB

In [ ]:
help(ir.datasets.iggytop)

In [ ]:
iggytop_tag = "data-2026.05.05.110331"
iggytop = ir.datasets.iggytop(tag=iggytop_tag)

In [ ]:
iggytop.obs["source"].value_counts()

In [ ]:
ir.pp.ir_dist(mdata, iggytop, metric="identity", sequence="aa")

In [ ]:
ir.tl.ir_query(
    mdata,
    iggytop,
    metric="identity",
    sequence="aa",
    receptor_arms="any",
    dual_ir="any",
)

In [ ]:
ir.tl.ir_query_annotate_df(
    mdata,
    iggytop,
    metric="identity",
    sequence="aa",
    include_ref_cols=["antigen_species", "antigen_name"],
)

In [ ]:
ir.tl.ir_query_annotate(
    mdata,
    iggytop,
    metric="identity",
    sequence="aa",
    include_ref_cols=["antigen_species"],
    strategy="most-frequent",
)

In [ ]:
mu.pl.embedding(mdata, "gex:umap", color="airr:antigen_species")

In [ ]:
ir.pp.ir_dist(
    mdata,
    metric="tcrdist",
    sequence="aa",
    cutoff=15,
)

## Table for every expanded clonotype

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Choose metadata columns
# -----------------------------

obs = mdata.obs.copy()

# Adjust these names if yours are different
clonotype_col = "airr:clone_id"
sample_col = "gex:sample_id"
condition_col = "gex:condition"
cluster_col = "gex:cell_annotation_05"

# IggyTop annotation columns
antigen_col = "airr:antigen_species"
antigen_name_col = "airr:antigen_name"


# Effector genes expected in mdata["gex"]
genes = ["Ifng", "Gzmb", "Prf1", "Nkg7", "Pdcd1", "Tox", "Cxcl13"]
genes = [g for g in genes if g in mdata["gex"].var_names]

# -----------------------------
# 2. Add gene expression to obs
# -----------------------------

expr = mdata["gex"][:, genes].to_df()
expr.columns = [f"{g}_expr" for g in genes]

obs = obs.join(expr)

# -----------------------------
# 3. Keep cells with clonotype info
# -----------------------------

obs_tcr = obs[
    obs[clonotype_col].notna()
].copy()

# -----------------------------
# 4. Per-clonotype summary
# -----------------------------

summary = (
    obs_tcr
    .groupby(clonotype_col)
    .agg(
        n_cells=(clonotype_col, "size"),
        n_samples=(sample_col, "nunique"),
        samples=(sample_col, lambda x: ",".join(sorted(x.astype(str).unique()))),
        conditions=(condition_col, lambda x: ",".join(sorted(x.astype(str).unique()))),
        major_cluster=(cluster_col, lambda x: x.value_counts().index[0]),
        antigen_species=(antigen_col, lambda x: ",".join(sorted(x.dropna().astype(str).unique()))),
       
    )
    .reset_index()
)

# Add mean expression per clonotype
expr_summary = (
    obs_tcr
    .groupby(clonotype_col)[[f"{g}_expr" for g in genes]]
    .mean()
    .reset_index()
)

summary = summary.merge(expr_summary, on=clonotype_col, how="left")

# -----------------------------
# 5. Add condition-specific abundance
# -----------------------------

clone_condition_counts = (
    obs_tcr
    .groupby([clonotype_col, condition_col])
    .size()
    .unstack(fill_value=0)
)

clone_condition_freq = clone_condition_counts.div(
    clone_condition_counts.sum(axis=0),
    axis=1
)

clone_condition_freq.columns = [
    f"freq_{c}" for c in clone_condition_freq.columns
]

summary = summary.merge(
    clone_condition_freq.reset_index(),
    on=clonotype_col,
    how="left"
)

# -----------------------------
# 6. Define priority categories
# -----------------------------

relevant_antigens = [
    "Tumor",
    "Tumor associated antigen",
    "Neoantigen",
    "Cancer",
    "Mus musculus",
    "Murine leukemia virus",
    "Lymphocytic choriomeningitis virus",
]

summary["relevant_iggytop_annotation"] = summary["antigen_species"].apply(
    lambda x: any(a.lower() in str(x).lower() for a in relevant_antigens)
)



summary = summary.sort_values(
    ["n_cells"],
    ascending=False
)

# -----------------------------
# 8. Save table
# -----------------------------

#summary.to_csv("prioritized_TCR_clonotypes_iggytop.csv", index=False)

In [ ]:
summary.head()

## Filter table to keep the clones that have more than 300 cells 

In [ ]:
summary_filtered = summary[
    summary["airr:clone_id"].isin(keep_clones)
].copy()

In [ ]:
summary_filtered

## 1. Clonotypes expanded specifically in effector + ICI mice

In [ ]:
summary_filtered = summary[
    summary["conditions"].isin(["effector"])
].copy()

In [ ]:
summary_filtered.antigen_species.unique()

In [ ]:
summary_filtered = summary_filtered[
    summary_filtered["antigen_species"].isin(["Neoantigen","Mus musculus"])]

In [ ]:
summary_filtered

They correspond to clonotypes with little cells

## 2. Clonotypes found in activated/exhausted CD8 populations

In [ ]:
summary_filtered = summary[
    summary["major_cluster"].isin(["Activated","Tissue resident memory","Cytotoxic/Effector memory"])
].copy()

In [ ]:
summary_filtered.head(10)

In [ ]:
#summary_filtered.antigen_species.unique()

## 3. Clonotypes annotated as mouse-derived

In [ ]:
summary_filtered = summary[
    summary["antigen_species"].isin(["Mus musculus"])
].copy()

In [ ]:
summary_filtered.head(10)

## 4. Tumor / Neoantigen annotations

In [ ]:
summary_filtered = summary[
    summary["antigen_species"].isin(["Tumor"])
].copy()

In [ ]:
summary_filtered.head(30)

## 5. Shared clonotypes across multiple mice

## Explore specific clonotypes

In [ ]:
ir.pp.ir_dist(
    mdata,
    metric="tcrdist",
    sequence="aa",
    cutoff=15,
)

In [ ]:
ir.tl.define_clonotype_clusters(mdata, sequence="aa", metric="tcrdist", receptor_arms="all", dual_ir="any")

In [ ]:
ir.tl.clonotype_network(mdata, min_cells=5, sequence="aa", metric="tcrdist")

### 7796 - related to species neoantigen

In [ ]:
with ir.get.airr_context(mdata, "junction_aa", ["VJ_1", "VDJ_1", "VJ_2", "VDJ_2"]):
    cdr3_ct_3390 = (
        # TODO astype(str) is required due to a bug in pandas ignoring `dropna=False`. It seems fixed in pandas 2.x
        mdata.obs.loc[lambda x: x["airr:cc_aa_tcrdist"] == "3390"]
        .astype(str)
        .groupby(
            [
                "VJ_1_junction_aa",
                "VDJ_1_junction_aa",
                "VJ_2_junction_aa",
                "VDJ_2_junction_aa",
                "airr:receptor_subtype",
            ],
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="n_cells")
    )
cdr3_ct_3390

In [ ]:
summary_filtered = summary[
    summary["samples"].isin(["effector1","efector2"])
].copy()

In [ ]:
#summary_filtered.head(30)

In [ ]:
summary_filtered = summary[
    summary["samples"].apply(
        lambda x: (
            "effector1" in str(x).split(",")
            and "effector2" in str(x).split(",")
        )
    )
].copy()

In [ ]:
summary_filtered.head()

## Conclusion


## Creating a file to have all the VDJ sequences from the top clones with > 10 cells 

In [ ]:
import pandas as pd
import scirpy as ir

mdata_clonotype_col = "airr:cc_aa_tcrdist"
keep_clones_str = [str(x) for x in keep_clones]

# -----------------------------
# 1. Make sure summary has clonotype as a column
# -----------------------------

summary2 = summary.copy()

if "airr:cc_aa_tcrdist" in summary2.columns:
    summary_clonotype_col = "airr:cc_aa_tcrdist"

elif "cc_aa_tcrdist" in summary2.columns:
    summary_clonotype_col = "cc_aa_tcrdist"

else:
    summary2 = summary2.reset_index()
    print(summary2.columns)
    
    # after reset_index, use the first column as clonotype column
    summary_clonotype_col = summary2.columns[0]

print("Using summary clonotype column:", summary_clonotype_col)

# -----------------------------
# 2. Extract CDR3 sequences for keep_clones
# -----------------------------

with ir.get.airr_context(
    mdata,
    "junction_aa",
    ["VJ_1", "VDJ_1", "VJ_2", "VDJ_2"]
):
    cdr3_keep_clones = (
        mdata.obs
        .loc[lambda x: x[mdata_clonotype_col].astype(str).isin(keep_clones_str)]
        .astype(str)
        .groupby(
            [
                mdata_clonotype_col,
                "VJ_1_junction_aa",
                "VDJ_1_junction_aa",
                "VJ_2_junction_aa",
                "VDJ_2_junction_aa",
                "airr:receptor_subtype",
            ],
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="n_cells_cdr3")
    )

cdr3_keep_clones = cdr3_keep_clones.rename(
    columns={mdata_clonotype_col: summary_clonotype_col}
)

# -----------------------------
# 3. Filter summary and merge
# -----------------------------
summary_keep_clones = summary2[
    summary2[summary_clonotype_col].astype(str).isin(keep_clones_str)
].copy()

summary_keep_clones[summary_clonotype_col] = (
    summary_keep_clones[summary_clonotype_col].astype(str)
)

cdr3_keep_clones[summary_clonotype_col] = (
    cdr3_keep_clones[summary_clonotype_col].astype(str)
)

merged_keep_clones = summary_keep_clones.merge(
    cdr3_keep_clones,
    on=summary_clonotype_col,
    how="left"
)

merged_keep_clones.to_csv(
    "prioritized_keep_clones_with_cdr3_sequences_normal.csv",
    index=False
)

merged_keep_clones.head()